# <br><center>**Lesson 00–12 – Debugging and Profiling in AI**

In [32]:
import torch

In [33]:
def debug_print(name, tensor):
    try:
        print(f"{name}: shape={tensor.shape}, dtype={tensor.dtype}, "
        f"device={tensor.device}, "
        f"min={tensor.min().item():.4f}, max={tensor.max().item():.4f}, "
        f"mean={tensor.mean().item():.4f}, median={tensor.median().item():.4f}, "
        f"has_nan={tensor.isnan().any().item()}, "
        f"has_inf={tensor.isinf().any().item()}")
    except:
        print(f"ERROR: debug-print failed for tensor {name}")

In [36]:
# test tensors
tensors = [
    torch.tensor(3.14),                                  # scalar
    torch.randn(3, 4),                                   # normal 2D
    torch.zeros(2, 3, 4),                                # zeros 3D
    torch.ones(5),                                       # ones 1D
    torch.arange(10, dtype=torch.int32),                 # int dtype # type: ignore
    torch.randn(2, 2).to(torch.float16),                 # fp16
    torch.randn(4, 4, device='cuda' if torch.cuda.is_available() else 'cpu'),
    torch.tensor([1.0, float('nan'), 3.0]),              # has NaN
    torch.tensor([1.0, float('inf'), -float('inf')]),    # has Inf
    torch.tensor([-1e10, 0.0, 1e10]),                    # extreme range
    torch.empty(0),                                      # empty (will crash min/max/median)
    torch.randn(1, 1, 1, 1),                             # 4D
    torch.tensor([[True, False], [False, True]]),        # bool (min/max may fail)
    torch.randn(100, 100) * 1e6,                         # large scale
    torch.randperm(10),                                  # int64 permutation
]

In [31]:
for idx, t in enumerate(tensors):
    debug_print(f"Tensor ({idx})", t)

Tensor (0): shape=torch.Size([]), dtype=torch.float32, device=cpu, min=3.1400, max=3.1400, mean=3.1400, median=3.1400, has_nan=False, has_inf=False
Tensor (1): shape=torch.Size([3, 4]), dtype=torch.float32, device=cpu, min=-1.3985, max=1.6447, mean=-0.0928, median=-0.3965, has_nan=False, has_inf=False
Tensor (2): shape=torch.Size([2, 3, 4]), dtype=torch.float32, device=cpu, min=0.0000, max=0.0000, mean=0.0000, median=0.0000, has_nan=False, has_inf=False
Tensor (3): shape=torch.Size([5]), dtype=torch.float32, device=cpu, min=1.0000, max=1.0000, mean=1.0000, median=1.0000, has_nan=False, has_inf=False
ERROR: debug-print failed for tensor Tensor (4)
Tensor (5): shape=torch.Size([2, 2]), dtype=torch.float16, device=cpu, min=-0.4441, max=2.1855, mean=0.2983, median=-0.4209, has_nan=False, has_inf=False
Tensor (6): shape=torch.Size([4, 4]), dtype=torch.float32, device=cpu, min=-0.9637, max=1.1344, mean=-0.0079, median=-0.2382, has_nan=False, has_inf=False
Tensor (7): shape=torch.Size([3]), d

### Example of smart breakpoint in pytorch training loop

```[python]
def training_step(model, batch, criterion, optimizer):
    inputs, labels = batch
    outputs = model(inputs)
    loss = criterion(outputs, labels)

    if loss.item() > 100 or torch.isnan(loss):
        breakpoint()

    loss.backward()
    optimizer.step()
```

after which following commands may be useful

```p outputs.shape ``` check output shapes <br>
```p loss.item() ``` check loss value <br>
```p torch.isnan(outputs).sum() ``` check number of nans in output <br>
```p model.fc1.weight.grad ``` check gradient <br>
```c ``` continue ```q ``` quit

In [ ]:
def training_step(model, batch, criterion, optimizer):
    inputs, labels = batch
    outputs = model(inputs)
    loss = criterion(outputs, labels)

    if loss.item() > 100 or torch.isnan(loss):
        breakpoint()

    loss.backward()
    optimizer.step()

In [59]:
# context
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- Tiny model ---
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)

# --- Fake dataset ---
X = torch.randn(64, 10)
y = torch.randint(0, 2, (64,))
loader = DataLoader(TensorDataset(X, y), batch_size=16)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [60]:
# --- Case 1: normal run (loss should be ~0.7, no breakpoint) ---
for batch in loader:
    training_step(model, batch, criterion, optimizer)

0.7057157754898071
0.6950086951255798
0.6967512965202332
0.6392069458961487


In [ ]:
# --- Case 2: trigger breakpoint via huge loss ---
# Scale up so CrossEntropyLoss blows past 100
bad_model = nn.Sequential(nn.Linear(10, 2))
with torch.no_grad():
    bad_model[0].weight.mul_(0)   # zero weights -> uniform logits
    bad_model[0].bias.copy_(torch.tensor([1000000.0, -1000000.0]))  # huge logit gap
bad_opt = torch.optim.SGD(bad_model.parameters(), lr=0.01)

batch = next(iter(loader))
training_step(bad_model, batch, criterion, bad_opt);   # hits breakpoint

1000000.0


In [ ]:
# --- Case 3: trigger breakpoint via NaN loss ---
with torch.no_grad():
    bad_model[0].bias.copy_(torch.tensor([float('nan'), 0.0]))
training_step(bad_model, batch, criterion, bad_opt);   # hits breakpoint

### 5 – cProfile and line_profiler

```[bash]
python -m cProfile -s cumtime train.py
```

This shows time of every function call. Instead for line by line profiling:

```[bash]
pip install line_profiler
```

```[python]
@profile
def train_step(model, data, target):
    output = model(data)
    loss = F.cross_entropy(output, target)
    loss.backward()
    return loss
```

### 6 – Memory Profiling

We may instead want to track the memory allocation of our cpu:

```[python]
...

if __name__ == "main":
    import tracemalloc

    tracemalloc.start()

    model = build_model()
    data = load_dataset()

    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics("lineno")
    for stat in top_stats[:10]:
        print(stat)
```

### Alternative (line-by-line)
Or you may use another memory profiler

```[python]
from memory_profiler import profile

@profile
def load_data():
    raw = read_csv("data.csv")
    processed = preprocess(raw)
    return processed
```

Run with $\texttt{python -m memory\_profiler script.py}$ to see memory allocation line-by-line.

### With PyTorch
You can also use torch itself:

$\texttt{}
import torch

if torch.cuda.is_available()
    print(torch.cuda.memory_summary())

    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
```

When you hit OOM (out of memory):

- Reduce batch size and try again
- Use $\texttt{torch.cuda.empty\_cache()}$ to free cached memory
- Use $\texttt{del tensor}$ followed by $\texttt{torch.cuda.epmty\_cache()}$ or $\texttt{gc.collect()}$ for large intermediate tensors
- Use mixed precision ($\texttt{torch.cuda.apm}$) to halve memory usage
- Use gradient checkpoints for very deep models

## Part 7 – Common AI bugs and how to catch them

### Shape mismatch

The most frequent bug is the model expecting shape $\texttt{[batch, channel, height, width]}$ but receiving $\texttt{[batch, features]}$.

```[python]
def check_shapes(model, sample_input):
    print(f"Input: {sample_input.shape}")
    hooks = []

    def make_hook(name)
        def hook(module, inp, out):
            in_shape = inp[0].shape if isinstance(inf, tuple) else inp.shape
            out_shape = out.shape if hasattr(out, "shape") else type(out)
            print(f"  {name}: {in_shape} ––> {out_shape}")
        return hook
    
    for name, module in model.named_modules():
        hooks.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        model(sample_input)
    
    for h in hooks:
        h.remove()
```

running this maps the shape transformation in your model.

### NaN Loss

NaN loss means something exploded. Common causes:

- Learning rate too high
- Division by zero in custom loss
- Log of zero or negative number
- Exploding gradients in RNNs

```[python]
def detect_nan(model, loss, step):
    if torch.isnan(loss):
        print(f"NaN loss at step {step}")
        for name, param in model.named_parameters():
            if param.grad is not None:
                if torch.isnan(param.grad).any():
                    print(f"  NaN gradient in {name}")
                if torch.isinf(param.grad).any():
                    print(f"  Inf gradient in {name}")
        return True
    return False
```

### Data Leakage

If your model does much better than expected (accuracy $\approx$ 99%, loss $\approx$ 0.00) then it is probably too good to be true...

```[python]
def check_data_leakage(train_set, test_set, idx_col="id"):
    train_ids = set(train_set[idx_col].to_list())
    test_ids = set(test_set[idx_col].to_list())
    overlap = train_ids & test_ids
    if overlap:
        print(f"DATA LEAKAGE (you stupid): {len(overlap)} samples in both train and test")
        return True
    return False
```

Also check for temporal leakage, training on future datapoints to predict the past.

### Wrong device

```[python]
def check_devices(model, *tensors):
    model_device = next(model.parameters()).device
    print(f"Model device: {model_device}")
    for i, t in enumerate(tensors):
        if t.device != model_device:
            print(f"  WARNING: tensor {i} on {t.device}, model on {model_device}")
```

## Part 8 – TensorBoard Basics
TensorBoard shows what's happening (lil o) inside training over time.

$\texttt{pip intsall tensorboard}$

```[python]
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/experiment_1")

for step in range(n_steps):
    loss = train_step(model, batch)

    writer.add_scaler("loss/train", loss.item(), step)
    writer.add_scaler("lr", optimizer.param_groups[0]["lr"], step)

    if step % 100 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(f"weights/{name}", param, step)
            if param.grad is not None:
                writer.add_histogram(f"gradients/{name}", param.grad, step)

writer.close()
```

Then launch with $\texttt{tensorboard --logdir=runs}$.

What to look for:

- Loss not decreasing: Learning rate too low, or model architecture issue
- Loss oscillating wildly: Learning rate too high
- Loss goes to NaN: Numerical instability (see NaN section above)
- Train loss decreasing, val loss increasing: Overfitting
- Weight histograms collapsing to zero: Vanishing gradients
- Gradient histograms exploding: Need gradient clipping

## Part 9 – VS Code debugger

For interactive debugging, configure VS Code with a $\texttt{launch.json}$:

```[json]
{
    "version": "0.2.0",
    "configurations": [
        {
            "name": "Debug Training",
            "type": "debugpy",
            "request": "launch",
            "program": "${file}",
            "console": "integratedTerminal",
            "justMyCode": false
        }
    ]
}
```

Set breakpoints by clicking the gutter. Use the Variables pane to inspect tensor properties. The Debug Console lets you run arbitrary Python expressions mid-execution.

Useful for stepping through data preprocessing pipelines where you want to see each transformation.

See **the missing semester lecture 4** on debugging.

## Use It

Here's the debugging workflow that catches most AI bugs:

- Before training: Run check_shapes with a sample batch. Verify input and output dimensions match expectations.
- First 10 steps: Use debug_print on loss, outputs, and gradients. Confirm nothing is NaN and values are in reasonable ranges.
- During training: Log loss, learning rate, and gradient norms. Use TensorBoard for visualization.
- When something breaks: Drop breakpoint() at the failure point. Inspect tensors interactively.
- For performance: Time your data loading vs forward vs backward pass. Profile memory if you're near OOM.